# RoBERTa Input Intervention Evaluation

This notebook evaluates the controlled input interventions created in `04.1_create_input_interventions.ipynb` using the previously trained RoBERTa classifier.

The objective is to measure how model predictions change when specific information sources are systematically modified:

1. the supplied target identity,
2. explicit candidate mentions in the retrieved context posts, and
3. label-correlated lexical cues.

All intervention datasets were constructed from the held-out human-annotated test set. Any data-driven intervention rules were derived exclusively from the training data before being applied to the test set.

The RoBERTa model is loaded in its previously trained and frozen state. Predictions on each modified input are compared with predictions on the corresponding original test example. For label-preserving interventions, changes in classification performance are also evaluated against the human gold labels. Target swapping is treated separately because the original stance label is no longer a valid gold label after changing the target.

This notebook is executed in Google Colab to make use of GPU acceleration. The project code, preprocessed datasets, intervention data, evaluation results, and trained RoBERTa model are stored in the public GitHub repository, with large model files tracked using Git LFS.

The workflow is:

- initialize Git LFS and clone the public repository,
- load the human-annotated test set and previously generated intervention variants,
- load the frozen fine-tuned RoBERTa model from `models/roberta`,
- reconstruct the model inputs using the same target-context representation as during training,
- run inference on the original and systematically modified inputs without retraining the model,
- compare classification performance, prediction changes, and class probabilities across intervention conditions, and
- save the resulting evaluation tables.

In [1]:
# GitHub + Git LFS setup

!apt-get update -qq
!apt-get install -y -qq git-lfs
!git lfs install

!git clone https://github.com/jakobtitz/political-stance-detection-analysis.git

%cd /content/political-stance-detection-analysis

%pip install -q -r requirements.txt

!git lfs pull

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package git-lfs.
(Reading database ... 122579 files and directories currently installed.)
Preparing to unpack .../git-lfs_3.0.2-1ubuntu0.3_amd64.deb ...
Unpacking git-lfs (3.0.2-1ubuntu0.3) ...
Setting up git-lfs (3.0.2-1ubuntu0.3) ...
Processing triggers for man-db (2.10.2-1) ...
Git LFS initialized.
Cloning into 'political-stance-detection-analysis'...
remote: Enumerating objects: 396, done.
remote: Counting objects: 100% (87/87), done.
remote: Compressing objects: 100% (68/68), done.
remote: Total 396 (delta 48), reused 34 (delta 19), pack-reused 309 (from 1)
Receiving objects: 100% (396/396), 9.77 MiB | 10.08 MiB/s, done.
Resolving deltas: 100% (155/155), done.
Filtering content: 100% (32/32), 790.45 MiB | 14.23 MiB/s, done.
/content/political-stance-detection-anal

---

### **1. Setup**

Load the libraries and define the paths to the fixed human test set, the frozen RoBERTa model, and the previously generated intervention datasets.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
)

from sklearn.metrics import classification_report, f1_score


DATA_DIR = Path("data/preprocessed")
INTERVENTION_DIR = Path("data/interventions")
RESULTS_DIR = Path("results/roberta_interventions")

HUMAN_TEST_PATH = DATA_DIR / "human_test.parquet"

MODEL_DIR = Path("models/roberta")

In [ ]:
# Collect paths for all intervention variants.
intervention_paths = {
    "target_masked": (
        INTERVENTION_DIR / "human_test_target_masked.parquet"
    ),
    "target_swapped": (
        INTERVENTION_DIR / "human_test_target_swapped.parquet"
    ),
    "candidate_mentions_masked": (
        INTERVENTION_DIR / "human_test_candidate_mentions_masked.parquet"
    ),
    "lexical_cues_top10_masked": (
        INTERVENTION_DIR / "human_test_lexical_cues_top10_masked.parquet"
    ),
    "lexical_cues_top25_masked": (
        INTERVENTION_DIR / "human_test_lexical_cues_top25_masked.parquet"
    )
}

for seed in range(1, 6):
    intervention_paths[
        f"candidate_mentions_control_seed{seed}"
    ] = (
        INTERVENTION_DIR
        / f"human_test_candidate_mentions_control_seed{seed}.parquet"
    )

    intervention_paths[
        f"lexical_cues_top10_control_seed{seed}"
    ] = (
        INTERVENTION_DIR
        / f"human_test_lexical_cues_top10_control_seed{seed}.parquet"
    )

    intervention_paths[
        f"lexical_cues_top25_control_seed{seed}"
    ] = (
        INTERVENTION_DIR
        / f"human_test_lexical_cues_top25_control_seed{seed}.parquet"
    )

---

### **Load the fixed test set and frozen model**

The original human-annotated test set provides the reference examples and gold labels. The frozen fine-tuned RoBERTa model and tokenizer are loaded from the cloned repository without modification and will be used for all original and intervention inputs.

In [ ]:
human_test = pd.read_parquet(HUMAN_TEST_PATH)

assert MODEL_DIR.exists(), f"Model directory not found: {MODEL_DIR.resolve()}"

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)

roberta = AutoModelForSequenceClassification.from_pretrained(
    MODEL_DIR
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

roberta.to(device)
roberta.eval()

print(f"Human test examples: {len(human_test):,}")
print(f"Columns: {human_test.columns.tolist()}")
print(f"Model directory: {MODEL_DIR.resolve()}")
print(f"Device: {device}")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Human test examples: 890
Columns: ['UserId', 'TargetEntity', 'StanceLabel', 'ContextPosts']
Model directory: /content/nlp-II-politiksky24/models/roberta
Device: cuda


---

### **2. Inspect the Frozen RoBERTa Model**

Before applying the model to the test interventions, inspect the loaded model and tokenizer and verify the label mapping used during training.

The intervention placeholder is also inspected under the frozen tokenizer to document how the synthetic masking token is represented by RoBERTa.

In [ ]:
print(f"Model type: {roberta.config.model_type}")
print(f"Number of labels: {roberta.config.num_labels}")
print(f"Maximum tokenizer length: {tokenizer.model_max_length}")
print(f"Separator token: {tokenizer.sep_token}")
print(f"Padding token: {tokenizer.pad_token}")
print(f"Model label mapping: {roberta.config.label2id}")

# Define the label mapping used throughout the evaluation
LABEL2ID = {
    "Against": 0,
    "Favor": 1,
    "Neither": 2,
}

ID2LABEL = {
    value: key
    for key, value in LABEL2ID.items()
}

LABEL_ORDER = ["Against", "Favor", "Neither"]

# Verify that the model's internal label order matches the expected label mapping
assert roberta.config.label2id == LABEL2ID
assert roberta.config.id2label == ID2LABEL

print(
    "Model label order:",
    [
        ID2LABEL[class_id]
        for class_id in range(roberta.config.num_labels)
    ],
)

Model type: roberta
Number of labels: 3
Maximum tokenizer length: 512
Separator token: </s>
Padding token: <pad>
Model label mapping: {'Against': 0, 'Favor': 1, 'Neither': 2}
Model label order: ['Against', 'Favor', 'Neither']


---

### **Verify the frozen context-masking token**

All context-level masking interventions were created with the synthetic token `requ`, which was selected and frozen before intervention generation in `04_select_masking_token.ipynb`.

For RoBERTa, the token is inspected under the frozen tokenizer to document how it is represented by the model. Target masking is treated separately and uses an empty target string rather than a synthetic token.

In [ ]:
CONTEXT_MASK_TOKEN = "requ"

mask_tokens = tokenizer.tokenize(CONTEXT_MASK_TOKEN)
mask_token_ids = tokenizer.encode(
    CONTEXT_MASK_TOKEN,
    add_special_tokens=False,
)

print("RoBERTa tokens:", mask_tokens)
print("Token IDs:", mask_token_ids)

RoBERTa tokens: ['requ']
Token IDs: [42172]


---

### **3. Load Intervention Datasets**

Load all previously generated intervention datasets. Each dataset contains the same held-out test examples in the same row order as the original test set, allowing predictions to be compared pairwise with the corresponding original input.

In [ ]:
intervention_data = {
    name: pd.read_parquet(path)
    for name, path in intervention_paths.items()
}

print(f"Loaded {len(intervention_data)} intervention datasets.")

for name in intervention_data:
    print(name)

Loaded 20 intervention datasets.
target_masked
target_swapped
candidate_mentions_masked
lexical_cues_top10_masked
lexical_cues_top25_masked
candidate_mentions_control_seed1
lexical_cues_top10_control_seed1
lexical_cues_top25_control_seed1
candidate_mentions_control_seed2
lexical_cues_top10_control_seed2
lexical_cues_top25_control_seed2
candidate_mentions_control_seed3
lexical_cues_top10_control_seed3
lexical_cues_top25_control_seed3
candidate_mentions_control_seed4
lexical_cues_top10_control_seed4
lexical_cues_top25_control_seed4
candidate_mentions_control_seed5
lexical_cues_top10_control_seed5
lexical_cues_top25_control_seed5


---

### **4. Construct Model Inputs**

Reconstruct the model inputs for the original test set and all intervention datasets using exactly the same input-construction function as during RoBERTa training.

Keeping the input construction unchanged ensures that any prediction differences are caused by the interventions rather than by differences in preprocessing or formatting.

In [ ]:
# Combine all valid context posts into the same context representation used during training
def build_context_text(context_posts):
    posts = [
        post["Content"]
        for post in context_posts
        if post["Content"] is not None
    ]

    return f" {tokenizer.sep_token} ".join(posts)


# Apply the same input-construction function to the original and all intervention variants
human_test["ContextText"] = (
    human_test["ContextPosts"].apply(build_context_text)
)

for name, df in intervention_data.items():
    df["ContextText"] = (
        df["ContextPosts"].apply(build_context_text)
    )

---

### **5. Generate Original Test Predictions**

Generate predictions and class probabilities for the unchanged human-annotated test set.

These outputs serve as the reference for all subsequent intervention comparisons. The frozen model is applied to the human-annotated test set without any refitting or adaptation.

In [ ]:
# Generate predictions and class probabilities in batches
def predict_roberta(df, batch_size=16):
    all_pred_ids = []
    all_probabilities = []

    targets = df["TargetEntity"].tolist()
    contexts = df["ContextText"].tolist()

    roberta.eval()

    with torch.inference_mode():
        for start in range(0, len(df), batch_size):
            end = start + batch_size

            batch_targets = targets[start:end]
            batch_contexts = contexts[start:end]

            encoded = tokenizer(
                batch_targets,
                batch_contexts,
                max_length=512,
                truncation="only_second",
                padding=True,
                return_tensors="pt",
            )

            encoded = {
                key: value.to(device)
                for key, value in encoded.items()
            }

            outputs = roberta(**encoded)

            probabilities = torch.softmax(
                outputs.logits,
                dim=-1,
            )

            pred_ids = torch.argmax(
                probabilities,
                dim=-1,
            )

            all_pred_ids.extend(
                pred_ids.cpu().numpy()
            )

            all_probabilities.append(
                probabilities.cpu().numpy()
            )

    return (
        np.array(all_pred_ids),
        np.concatenate(all_probabilities, axis=0),
    )

In [ ]:
original_pred_ids, original_probabilities = predict_roberta(
    human_test
)

original_predictions = np.array([
    ID2LABEL[pred_id]
    for pred_id in original_pred_ids
])

print(f"Predictions: {len(original_predictions):,}")
print(f"Probability matrix: {original_probabilities.shape}")

Predictions: 890
Probability matrix: (890, 3)


---

### **6. Evaluate Original Test Performance**

Evaluate the unchanged human-annotated test set to establish the RoBERTa reference performance.

Macro-F1 is used as the main overall metric because the stance classes are imbalanced. Class-specific precision, recall, and F1-scores are additionally reported to inspect performance differences between stance classes.

In [ ]:
y_true = human_test["StanceLabel"].to_numpy()

original_macro_f1 = f1_score(
    y_true,
    original_predictions,
    labels=LABEL_ORDER,
    average="macro",
)

print(f"Original Macro-F1: {original_macro_f1:.4f}")

Original Macro-F1: 0.8123


In [ ]:
# Compute the reference classification report for the unmodified test set
original_report = classification_report(
    y_true,
    original_predictions,
    labels=LABEL_ORDER,
    output_dict=True,
    zero_division=0,
)

original_report = pd.DataFrame(original_report).T

original_report

,precision,recall,f1-score,support
Against,0.911308,0.934091,0.922559,440.000000
Favor,0.800885,0.770213,0.785249,235.000000
Neither,0.732394,0.725581,0.728972,215.000000
accuracy,0.840449,0.840449,0.840449,0.840449
macro avg,0.814863,0.809962,0.812260,890.000000
weighted avg,0.838931,0.840449,0.839538,890.000000


The frozen RoBERTa model achieves a Macro-F1 of 0.812 and an accuracy of 0.840 on the unmodified human-annotated test set. Performance is strongest for Against (F1 = 0.923), followed by Favor (0.785), while Neither is the most difficult class (0.729).

---

### **7. Generate Intervention Predictions**

Apply the same frozen RoBERTa model to every intervention dataset.

For each intervention, store both the predicted stance labels and the corresponding class probabilities. These outputs will later be compared pairwise with the predictions on the unchanged test inputs.

In [ ]:
intervention_predictions = {}
intervention_probabilities = {}

# Apply the frozen RoBERTa model to all intervention variants
for name, df in intervention_data.items():

    pred_ids, probabilities = predict_roberta(df)

    predictions = np.array([
        ID2LABEL[pred_id]
        for pred_id in pred_ids
    ])

    intervention_predictions[name] = predictions
    intervention_probabilities[name] = probabilities

---

### **8. Compare Intervention Outcomes**

Compare each intervention with the predictions on the unchanged test inputs.

For label-preserving interventions, Macro-F1 is calculated against the human gold labels and its change relative to the original performance is reported. The prediction flip rate measures the proportion of examples for which the predicted stance label changes after the intervention.

Target swapping is evaluated only through prediction changes because changing the target invalidates the original gold stance label.

In [ ]:
# Compare each intervention with the original predictions using Macro-F1 change and prediction flip rate
comparison_rows = [
    {
        "condition": "original",
        "macro_f1": original_macro_f1,
        "delta_macro_f1": 0.0,
        "flip_rate": 0.0,
    }
]

for name, predictions in intervention_predictions.items():

    flip_rate = np.mean(
        predictions != original_predictions
    )

    # Target swapping has no valid gold labels
    if name == "target_swapped":
        macro_f1 = np.nan
        delta_macro_f1 = np.nan

    else:
        macro_f1 = f1_score(
            y_true,
            predictions,
            labels=LABEL_ORDER,
            average="macro",
        )

        delta_macro_f1 = (
            macro_f1 - original_macro_f1
        )

    comparison_rows.append(
        {
            "condition": name,
            "macro_f1": macro_f1,
            "delta_macro_f1": delta_macro_f1,
            "flip_rate": flip_rate,
        }
    )

comparison_results = pd.DataFrame(
    comparison_rows
)

comparison_results

,condition,macro_f1,delta_macro_f1,flip_rate
0,original,0.812260,0.000000,0.000000
1,target_masked,0.742188,-0.070072,0.113483
2,target_swapped,NaN,NaN,0.715730
3,candidate_mentions_masked,0.554192,-0.258068,0.389888
4,lexical_cues_top10_masked,0.814043,0.001783,0.004494
5,lexical_cues_top25_masked,0.812818,0.000558,0.005618
6,candidate_mentions_control_seed1,0.804579,-0.007682,0.037079
7,lexical_cues_top10_control_seed1,0.813232,0.000972,0.001124
8,lexical_cues_top25_control_seed1,0.810291,-0.001969,0.005618
9,candidate_mentions_control_seed2,0.804225,-0.008035,0.042697


The RoBERTa classifier shows substantial sensitivity to both the supplied target and explicit candidate mentions in the retrieved context. Target masking reduces Macro-F1 by approximately 0.070 and changes 11.3% of predictions, while target swapping changes 71.6% of predictions. Candidate-mention masking has an even stronger effect, reducing Macro-F1 by approximately 0.258 and changing 39.0% of predictions. In contrast, masking the identified top-10 or top-25 lexical cues produces almost no change in performance or predicted labels.

---

### **9. Compare Masking Interventions with Matched Controls**

Aggregate the five matched-control runs for each masking intervention.

The control conditions replace approximately the same number of tokens as the corresponding masking intervention, but at matched non-target locations. Comparing the actual intervention with the average control effect helps distinguish reliance on the selected information from the general effect of modifying the same amount of input text.

In [ ]:
control_groups = {
    "candidate_mentions_masked": "candidate_mentions_control_",
    "lexical_cues_top10_masked": "lexical_cues_top10_control_",
    "lexical_cues_top25_masked": "lexical_cues_top25_control_",
}

control_comparison_rows = []

# Compare each masking intervention with its five matched controls
for intervention, control_prefix in control_groups.items():

    intervention_row = comparison_results.loc[
        comparison_results["condition"] == intervention
    ].iloc[0]

    controls = comparison_results[
        comparison_results["condition"].str.startswith(
            control_prefix
        )
    ]

    assert len(controls) == 5

    control_comparison_rows.append(
        {
            "intervention": intervention,
            "intervention_macro_f1": intervention_row["macro_f1"],
            "control_macro_f1_mean": controls["macro_f1"].mean(),
            "control_macro_f1_sd": controls["macro_f1"].std(),
            "macro_f1_vs_control": (
                intervention_row["macro_f1"]
                - controls["macro_f1"].mean()
            ),
            "intervention_flip_rate": intervention_row["flip_rate"],
            "control_flip_rate_mean": controls["flip_rate"].mean(),
            "control_flip_rate_sd": controls["flip_rate"].std(),
            "flip_rate_vs_control": (
                intervention_row["flip_rate"]
                - controls["flip_rate"].mean()
            ),
        }
    )

control_comparison = pd.DataFrame(
    control_comparison_rows
)

control_comparison

,intervention,intervention_macro_f1,control_macro_f1_mean,control_macro_f1_sd,macro_f1_vs_control,intervention_flip_rate,control_flip_rate_mean,control_flip_rate_sd,flip_rate_vs_control
0,candidate_mentions_masked,0.554192,0.807343,0.005085,-0.253151,0.389888,0.035955,0.004424,0.353933
1,lexical_cues_top10_masked,0.814043,0.812832,0.000420,0.001212,0.004494,0.001798,0.000615,0.002697
2,lexical_cues_top25_masked,0.812818,0.811984,0.001388,0.000834,0.005618,0.004270,0.001846,0.001348


Masking explicit candidate mentions produces a substantially stronger effect than matched random removals. Macro-F1 decreases to 0.554 compared with an average of 0.807 under the controls, while the prediction flip rate increases from approximately 3.6% to 39.0%. In contrast, masking the top-10 and top-25 label-associated lexical cues produces performance and prediction changes that remain very close to their matched controls.

---

### **10. Analyze Changes in Predicted Probabilities**

Prediction flips capture only cases in which an intervention changes the final predicted stance label. Smaller changes in model confidence can occur even when the predicted label remains unchanged.

For each intervention, we therefore measure:

- the mean change in probability assigned to the originally predicted class, and
- the mean absolute change across all three class probabilities.

Negative changes in the original-class probability indicate that the intervention weakens the model's original decision.

In [ ]:
original_pred_indices = np.array([
    LABEL2ID[label]
    for label in original_predictions
])

row_indices = np.arange(len(human_test))

original_predicted_class_prob = original_probabilities[
    row_indices,
    original_pred_indices,
]

probability_rows = []

# Measure how each intervention changes the predicted probabilities relative to the original inputs
for name, probabilities in intervention_probabilities.items():

    intervention_original_class_prob = probabilities[
        row_indices,
        original_pred_indices,
    ]

    original_class_probability_change = (
        intervention_original_class_prob
        - original_predicted_class_prob
    )

    mean_absolute_probability_change = np.abs(
        probabilities - original_probabilities
    ).mean()

    probability_rows.append(
        {
            "condition": name,
            "mean_original_class_probability_change":
                original_class_probability_change.mean(),
            "mean_absolute_probability_change":
                mean_absolute_probability_change,
        }
    )

probability_results = pd.DataFrame(probability_rows)

probability_results

,condition,mean_original_class_probability_change,mean_absolute_probability_change
0,target_masked,-0.078217,0.072664
1,target_swapped,-0.622788,0.443337
2,candidate_mentions_masked,-0.336434,0.258036
3,lexical_cues_top10_masked,-0.001925,0.001942
4,lexical_cues_top25_masked,-0.001395,0.003780
5,candidate_mentions_control_seed1,-0.017865,0.021953
6,lexical_cues_top10_control_seed1,-0.000056,0.001017
7,lexical_cues_top25_control_seed1,-0.001826,0.002682
8,candidate_mentions_control_seed2,-0.019963,0.022105
9,lexical_cues_top10_control_seed2,-0.000338,0.000943


In [ ]:
probability_control_rows = []

# Compare probability changes from each masking intervention with its matched controls
for intervention, control_prefix in control_groups.items():

    intervention_row = probability_results.loc[
        probability_results["condition"] == intervention
    ].iloc[0]

    controls = probability_results[
        probability_results["condition"].str.startswith(
            control_prefix
        )
    ]

    assert len(controls) == 5

    probability_control_rows.append(
        {
            "intervention": intervention,

            "intervention_original_class_prob_change":
                intervention_row[
                    "mean_original_class_probability_change"
                ],

            "control_original_class_prob_change_mean":
                controls[
                    "mean_original_class_probability_change"
                ].mean(),

            "control_original_class_prob_change_sd":
                controls[
                    "mean_original_class_probability_change"
                ].std(),

            "intervention_mean_abs_prob_change":
                intervention_row[
                    "mean_absolute_probability_change"
                ],

            "control_mean_abs_prob_change_mean":
                controls[
                    "mean_absolute_probability_change"
                ].mean(),

            "control_mean_abs_prob_change_sd":
                controls[
                    "mean_absolute_probability_change"
                ].std(),
        }
    )

probability_control_comparison = pd.DataFrame(
    probability_control_rows
)

probability_control_comparison

,intervention,intervention_original_class_prob_change,control_original_class_prob_change_mean,control_original_class_prob_change_sd,intervention_mean_abs_prob_change,control_mean_abs_prob_change_mean,control_mean_abs_prob_change_sd
0,candidate_mentions_masked,-0.336434,-0.018844,0.001095,0.258036,0.022342,0.000548
1,lexical_cues_top10_masked,-0.001925,-0.000753,0.000625,0.001942,0.001270,0.000354
2,lexical_cues_top25_masked,-0.001395,-0.001875,0.000388,0.003780,0.002727,0.000381


Probability-based results reinforce the performance-based findings. Target swapping produces the strongest change in RoBERTa's predicted probability distributions, while masking explicit candidate mentions also substantially reduces the probability assigned to the model's original prediction. Matched candidate-control removals produce only minor changes. In contrast, masking the top-10 and top-25 lexical cue sets results in only small probability shifts that remain close to the corresponding controls.

---

### **11. Analyze Intervention Effects by Target**

Evaluate whether intervention effects differ between the two political targets.

All examples are grouped by their original target in the human-annotated test set. For label-preserving interventions, target-specific Macro-F1 and prediction flip rates are reported. For target swapping, only behavioral changes are evaluated because the original gold label is no longer valid after changing the target.

The preceding dataset analysis showed a strong association between target and stance label in both the training and human-annotated test data. Target-specific Macro-F1 is therefore retained as a performance measure, but absolute Macro-F1 values are not interpreted as directly comparable measures of difficulty across targets. The main focus is on within-target changes relative to the original predictions.

In [ ]:
target_rows = []

# Recompute intervention effects separately for each target to assess whether model sensitivity differs between Trump and Harris
for target in human_test["TargetEntity"].unique():

    target_mask = (
        human_test["TargetEntity"].to_numpy() == target
    )

    target_indices = np.where(target_mask)[0]

    target_y_true = y_true[target_mask]
    target_original_predictions = original_predictions[target_mask]

    original_target_macro_f1 = f1_score(
        target_y_true,
        target_original_predictions,
        labels=LABEL_ORDER,
        average="macro",
    )

    original_target_class_prob = original_probabilities[
        target_indices,
        original_pred_indices[target_indices],
    ]

    target_rows.append(
        {
            "target": target,
            "condition": "original",
            "macro_f1": original_target_macro_f1,
            "delta_macro_f1": 0.0,
            "flip_rate": 0.0,
            "mean_original_class_probability_change": 0.0,
            "mean_absolute_probability_change": 0.0,
        }
    )

    for name, predictions in intervention_predictions.items():

        target_predictions = predictions[target_mask]

        target_probabilities = intervention_probabilities[name][
            target_indices
        ]

        flip_rate = np.mean(
            target_predictions
            != target_original_predictions
        )

        intervention_target_class_prob = target_probabilities[
            np.arange(len(target_indices)),
            original_pred_indices[target_indices],
        ]

        mean_original_class_probability_change = (
            intervention_target_class_prob
            - original_target_class_prob
        ).mean()

        mean_absolute_probability_change = np.abs(
            target_probabilities
            - original_probabilities[target_indices]
        ).mean()

        if name == "target_swapped":
            macro_f1 = np.nan
            delta_macro_f1 = np.nan

        else:
            macro_f1 = f1_score(
                target_y_true,
                target_predictions,
                labels=LABEL_ORDER,
                average="macro",
            )

            delta_macro_f1 = (
                macro_f1
                - original_target_macro_f1
            )

        target_rows.append(
            {
                "target": target,
                "condition": name,
                "macro_f1": macro_f1,
                "delta_macro_f1": delta_macro_f1,
                "flip_rate": flip_rate,
                "mean_original_class_probability_change":
                    mean_original_class_probability_change,
                "mean_absolute_probability_change":
                    mean_absolute_probability_change,
            }
        )

target_results = pd.DataFrame(target_rows)

In [ ]:
main_conditions = [
    "original",
    "target_masked",
    "target_swapped",
    "candidate_mentions_masked",
    "lexical_cues_top10_masked",
    "lexical_cues_top25_masked",
]

target_results[
    target_results["condition"].isin(main_conditions)
].reset_index(drop=True)

,target,condition,macro_f1,delta_macro_f1,flip_rate,mean_original_class_probability_change,mean_absolute_probability_change
0,Trump,original,0.688770,0.000000,0.000000,0.000000,0.000000
1,Trump,target_masked,0.654556,-0.034214,0.092135,-0.070757,0.057297
2,Trump,target_swapped,NaN,NaN,0.914607,-0.816437,0.557189
3,Trump,candidate_mentions_masked,0.601696,-0.087074,0.357303,-0.353563,0.247645
4,Trump,lexical_cues_top10_masked,0.693766,0.004996,0.002247,-0.002035,0.001937
5,Trump,lexical_cues_top25_masked,0.693766,0.004996,0.002247,-0.000244,0.004034
6,Harris,original,0.728334,0.000000,0.000000,0.000000,0.000000
7,Harris,target_masked,0.622490,-0.105844,0.134831,-0.085678,0.088031
8,Harris,target_swapped,NaN,NaN,0.516854,-0.429139,0.329486
9,Harris,candidate_mentions_masked,0.429207,-0.299128,0.422472,-0.319305,0.268428


---

### **12. Analyze Intervention Effects by Stance Class**

Examine whether intervention effects differ across the three stance classes.

For each label-preserving condition, precision, recall, and F1-score are calculated separately for `Against`, `Favor`, and `Neither`. Changes in class-specific F1 relative to the unchanged test inputs are reported to determine whether global performance changes are concentrated in particular stance classes.

Target swapping is excluded from this analysis because changing the target invalidates the original gold stance label.

In [ ]:
class_rows = []

# Compute class-specific precision, recall, F1, and support for all label-preserving conditions
label_preserving_predictions = {
    "original": original_predictions,
    **{
        name: predictions
        for name, predictions in intervention_predictions.items()
        if name != "target_swapped"
    },
}

for condition, predictions in label_preserving_predictions.items():

    report = classification_report(
        y_true,
        predictions,
        labels=LABEL_ORDER,
        output_dict=True,
        zero_division=0,
    )

    for label in LABEL_ORDER:

        class_rows.append(
            {
                "condition": condition,
                "class": label,
                "precision": report[label]["precision"],
                "recall": report[label]["recall"],
                "f1": report[label]["f1-score"],
                "support": report[label]["support"],
            }
        )

class_results = pd.DataFrame(class_rows)

In [ ]:
# Compute class-specific F1 changes relative to the original predictions
original_class_f1 = (
    class_results[
        class_results["condition"] == "original"
    ]
    .set_index("class")["f1"]
    .to_dict()
)

class_results["delta_f1"] = class_results.apply(
    lambda row: (
        row["f1"] - original_class_f1[row["class"]]
    ),
    axis=1,
)

In [ ]:
# Display only the main label-preserving interventions
main_label_preserving_conditions = [
    "original",
    "target_masked",
    "candidate_mentions_masked",
    "lexical_cues_top10_masked",
    "lexical_cues_top25_masked",
]

class_results[
    class_results["condition"].isin(
        main_label_preserving_conditions
    )
].reset_index(drop=True)

,condition,class,precision,recall,f1,support,delta_f1
0,original,Against,0.911308,0.934091,0.922559,440.0,0.000000
1,original,Favor,0.800885,0.770213,0.785249,235.0,0.000000
2,original,Neither,0.732394,0.725581,0.728972,215.0,0.000000
3,target_masked,Against,0.840708,0.863636,0.852018,440.0,-0.070541
4,target_masked,Favor,0.774775,0.731915,0.752735,235.0,-0.032514
5,target_masked,Neither,0.620370,0.623256,0.621810,215.0,-0.107162
6,candidate_mentions_masked,Against,0.963504,0.600000,0.739496,440.0,-0.183063
7,candidate_mentions_masked,Favor,0.876923,0.242553,0.380000,235.0,-0.405249
8,candidate_mentions_masked,Neither,0.377495,0.967442,0.543081,215.0,-0.185891
9,lexical_cues_top10_masked,Against,0.909292,0.934091,0.921525,440.0,-0.001034


The class-specific results show that candidate-mention masking has the strongest effect on the **Favor** class, whose F1-score drops from 0.785 to 0.380, primarily due to a substantial decrease in recall. Performance also declines for **Against** and **Neither**. Notably, candidate masking strongly increases recall for Neither while reducing its precision, indicating that the model frequently shifts predictions toward the neutral class after explicit candidate references are removed. Target masking causes smaller reductions across all three classes, while the lexical-cue interventions again have almost no effect.

---

### **13. Audit Intervention Coverage**

Quantify how frequently each masking intervention actually modifies the human-annotated test inputs.

A small behavioral effect can only be interpreted as weak model reliance if the corresponding intervention affects a meaningful number of test examples. We therefore report the number and proportion of examples whose model input changes under each intervention.

In [ ]:
# Measure intervention coverage by checking whether either part of the RoBERTa input changes
coverage_conditions = [
    "target_masked",
    "candidate_mentions_masked",
    "lexical_cues_top10_masked",
    "lexical_cues_top25_masked",
]

coverage_rows = []

original_targets = human_test["TargetEntity"].to_numpy()
original_contexts = human_test["ContextText"].to_numpy()

for name in coverage_conditions:

    intervention_targets = (
        intervention_data[name]["TargetEntity"].to_numpy()
    )

    intervention_contexts = (
        intervention_data[name]["ContextText"].to_numpy()
    )

    changed = (
        (intervention_targets != original_targets)
        | (intervention_contexts != original_contexts)
    )

    coverage_rows.append(
        {
            "condition": name,
            "changed_examples": changed.sum(),
            "total_examples": len(changed),
            "coverage_rate": changed.mean(),
        }
    )

coverage_results = pd.DataFrame(coverage_rows)

coverage_results

,condition,changed_examples,total_examples,coverage_rate
0,target_masked,890,890,1.000000
1,candidate_mentions_masked,843,890,0.947191
2,lexical_cues_top10_masked,126,890,0.141573
3,lexical_cues_top25_masked,275,890,0.308989


---

### **14. Analyze Effects Among Modified Examples**

The preceding analyses report behavioral changes across the complete test set. Because intervention coverage differs substantially between conditions, these aggregate effects can be diluted by examples that were not modified.

We therefore additionally report prediction flips and probability changes conditional on examples whose model input was actually changed. These conditional measures complement, rather than replace, the full-test results.

In [ ]:
affected_rows = []

# Recompute intervention effects only for examples whose model input was actually changed by the corresponding intervention
for name in coverage_conditions:

    intervention_targets = (
        intervention_data[name]["TargetEntity"].to_numpy()
    )

    intervention_contexts = (
        intervention_data[name]["ContextText"].to_numpy()
    )

    changed = (
        (intervention_targets != original_targets)
        | (intervention_contexts != original_contexts)
    )

    predictions = intervention_predictions[name]
    probabilities = intervention_probabilities[name]

    affected_indices = np.where(changed)[0]

    affected_flip_rate = np.mean(
        predictions[changed]
        != original_predictions[changed]
    )

    affected_original_class_prob_change = (
        probabilities[
            affected_indices,
            original_pred_indices[affected_indices],
        ]
        - original_probabilities[
            affected_indices,
            original_pred_indices[affected_indices],
        ]
    ).mean()

    affected_mean_abs_prob_change = np.abs(
        probabilities[affected_indices]
        - original_probabilities[affected_indices]
    ).mean()

    affected_rows.append(
        {
            "condition": name,
            "affected_examples": changed.sum(),
            "affected_flip_rate": affected_flip_rate,
            "affected_original_class_prob_change":
                affected_original_class_prob_change,
            "affected_mean_abs_prob_change":
                affected_mean_abs_prob_change,
        }
    )

affected_results = pd.DataFrame(
    affected_rows
)

affected_results

,condition,affected_examples,affected_flip_rate,affected_original_class_prob_change,affected_mean_abs_prob_change
0,target_masked,890,0.113483,-0.078217,0.072664
1,candidate_mentions_masked,843,0.411625,-0.355191,0.272423
2,lexical_cues_top10_masked,126,0.031746,-0.013598,0.013717
3,lexical_cues_top25_masked,275,0.018182,-0.004516,0.012234


Among examples that were actually modified, candidate-mention masking produces by far the strongest effect, changing 41.2% of predictions and reducing the probability of the originally predicted class by about 0.355 on average. Target masking has a smaller but still noticeable effect. In contrast, the lexical-cue interventions remain weak even when restricting the analysis to affected examples, suggesting that their small overall effects are not simply caused by limited intervention coverage.

---

### **15. Compare Conditional Effects with Matched Controls**

Compare each masking intervention with its matched-control variants among examples whose model input was actually modified.

Because intervention coverage differs substantially across intervention types, behavioral effects are calculated conditionally on the affected examples. The matched controls provide a like-for-like comparison with random modifications of comparable extent, and the five control runs are summarized by their mean and standard deviation.

In [ ]:
conditional_rows = []

conditional_conditions = [
    name
    for name in intervention_predictions
    if (
        name == "candidate_mentions_masked"
        or name == "lexical_cues_top10_masked"
        or name == "lexical_cues_top25_masked"
        or "_control_seed" in name
    )
]

# Compute conditional effects for the masking interventions and all matched controls
for name in conditional_conditions:

    intervention_targets = (
        intervention_data[name]["TargetEntity"].to_numpy()
    )

    intervention_contexts = (
        intervention_data[name]["ContextText"].to_numpy()
    )

    changed = (
        (intervention_targets != original_targets)
        | (intervention_contexts != original_contexts)
    )

    affected_indices = np.where(changed)[0]

    predictions = intervention_predictions[name]
    probabilities = intervention_probabilities[name]

    flip_rate = np.mean(
        predictions[affected_indices]
        != original_predictions[affected_indices]
    )

    original_class_prob_change = (
        probabilities[
            affected_indices,
            original_pred_indices[affected_indices],
        ]
        - original_probabilities[
            affected_indices,
            original_pred_indices[affected_indices],
        ]
    ).mean()

    mean_abs_prob_change = np.abs(
        probabilities[affected_indices]
        - original_probabilities[affected_indices]
    ).mean()

    conditional_rows.append(
        {
            "condition": name,
            "affected_examples": len(affected_indices),
            "flip_rate": flip_rate,
            "mean_original_class_probability_change":
                original_class_prob_change,
            "mean_absolute_probability_change":
                mean_abs_prob_change,
        }
    )

conditional_results = pd.DataFrame(
    conditional_rows
)

conditional_results

,condition,affected_examples,flip_rate,mean_original_class_probability_change,mean_absolute_probability_change
0,candidate_mentions_masked,843,0.411625,-0.355191,0.272423
1,lexical_cues_top10_masked,126,0.031746,-0.013598,0.013717
2,lexical_cues_top25_masked,275,0.018182,-0.004516,0.012234
3,candidate_mentions_control_seed1,843,0.039146,-0.018861,0.023177
4,lexical_cues_top10_control_seed1,126,0.007937,-0.000393,0.007186
5,lexical_cues_top25_control_seed1,275,0.018182,-0.005911,0.008681
6,candidate_mentions_control_seed2,843,0.045077,-0.021076,0.023338
7,lexical_cues_top10_control_seed2,126,0.007937,-0.002388,0.006662
8,lexical_cues_top25_control_seed2,275,0.010909,-0.006103,0.008180
9,candidate_mentions_control_seed3,843,0.037960,-0.018800,0.024510


In [ ]:
conditional_control_rows = []

# Aggregate the five matched controls for each intervention
for intervention, control_prefix in control_groups.items():

    intervention_row = conditional_results.loc[
        conditional_results["condition"] == intervention
    ].iloc[0]

    controls = conditional_results[
        conditional_results["condition"].str.startswith(
            control_prefix
        )
    ]

    assert len(controls) == 5

    conditional_control_rows.append(
        {
            "intervention": intervention,

            "intervention_flip_rate":
                intervention_row["flip_rate"],

            "control_flip_rate_mean":
                controls["flip_rate"].mean(),

            "control_flip_rate_sd":
                controls["flip_rate"].std(),

            "intervention_original_class_prob_change":
                intervention_row[
                    "mean_original_class_probability_change"
                ],

            "control_original_class_prob_change_mean":
                controls[
                    "mean_original_class_probability_change"
                ].mean(),

            "control_original_class_prob_change_sd":
                controls[
                    "mean_original_class_probability_change"
                ].std(),

            "intervention_mean_abs_prob_change":
                intervention_row[
                    "mean_absolute_probability_change"
                ],

            "control_mean_abs_prob_change_mean":
                controls[
                    "mean_absolute_probability_change"
                ].mean(),

            "control_mean_abs_prob_change_sd":
                controls[
                    "mean_absolute_probability_change"
                ].std(),
        }
    )

conditional_control_comparison = pd.DataFrame(
    conditional_control_rows
)

conditional_control_comparison

,intervention,intervention_flip_rate,control_flip_rate_mean,control_flip_rate_sd,intervention_original_class_prob_change,control_original_class_prob_change_mean,control_original_class_prob_change_sd,intervention_mean_abs_prob_change,control_mean_abs_prob_change_mean,control_mean_abs_prob_change_sd
0,candidate_mentions_masked,0.411625,0.037960,0.004670,-0.355191,-0.019895,0.001156,0.272423,0.023588,0.000578
1,lexical_cues_top10_masked,0.031746,0.012698,0.004347,-0.013598,-0.005321,0.004412,0.013717,0.008968,0.002497
2,lexical_cues_top25_masked,0.018182,0.013818,0.005975,-0.004516,-0.006069,0.001255,0.012234,0.008827,0.001234


Conditional comparison with the matched controls confirms that candidate-mention masking has a strong specific effect beyond general text removal. Among affected examples, candidate masking changes 41.2% of predictions compared with only 3.8% for the matched controls and causes substantially larger probability shifts. The lexical-cue interventions show only small additional effects over their controls, with the top-25 condition being particularly close to the matched-control baseline.

---

### **16. Analyze Correctness Transitions**

Prediction flips do not indicate whether an intervention harms or improves a previously made decision.

For each label-preserving main intervention, we therefore distinguish between predictions that remain correct, change from correct to incorrect, change from incorrect to correct, or remain incorrect relative to the human gold labels.

In [ ]:
transition_conditions = [
    "target_masked",
    "candidate_mentions_masked",
    "lexical_cues_top10_masked",
    "lexical_cues_top25_masked",
]

original_correct = (
    original_predictions == y_true
)

transition_rows = []

# Analyze how each intervention changes prediction correctness
for name in transition_conditions:

    intervention_correct = (
        intervention_predictions[name] == y_true
    )

    stable_correct = (
        original_correct & intervention_correct
    )

    correct_to_wrong = (
        original_correct & ~intervention_correct
    )

    wrong_to_correct = (
        ~original_correct & intervention_correct
    )

    stable_wrong = (
        ~original_correct & ~intervention_correct
    )

    transition_rows.append(
        {
            "condition": name,

            "stable_correct_n":
                stable_correct.sum(),

            "correct_to_wrong_n":
                correct_to_wrong.sum(),

            "wrong_to_correct_n":
                wrong_to_correct.sum(),

            "stable_wrong_n":
                stable_wrong.sum(),

            "correct_to_wrong_rate":
                correct_to_wrong.mean(),

            "wrong_to_correct_rate":
                wrong_to_correct.mean(),

            "net_correct_change":
                wrong_to_correct.sum()
                - correct_to_wrong.sum(),
        }
    )

transition_results = pd.DataFrame(
    transition_rows
)

transition_results

,condition,stable_correct_n,correct_to_wrong_n,wrong_to_correct_n,stable_wrong_n,correct_to_wrong_rate,wrong_to_correct_rate,net_correct_change
0,target_masked,671,77,15,127,0.086517,0.016854,-62
1,candidate_mentions_masked,474,274,55,87,0.307865,0.061798,-219
2,lexical_cues_top10_masked,747,1,2,140,0.001124,0.002247,1
3,lexical_cues_top25_masked,746,2,2,140,0.002247,0.002247,0


Target masking produces a net loss of 62 correct predictions, while candidate-mention masking has a substantially stronger effect with a net loss of 219 correct predictions. In contrast, the lexical-cue interventions produce virtually no net change in correctness. This further indicates that RoBERTa relies strongly on explicit candidate information, whereas the selected label-associated lexical cues play only a minor role.

---

### **17. Inspect Prediction Transitions under Candidate Masking**

Candidate-mention masking produced by far the strongest behavioral and performance effect. As a descriptive follow-up analysis, we therefore examine this intervention in more detail by comparing the original predicted labels with the predictions obtained after masking explicit candidate references.

The transition matrix shows which predicted stance classes are preserved and which classes the model switches to after candidate information is removed.

In [ ]:
# Count how predictions transition between stance classes after masking candidate mentions
candidate_transition_counts = pd.crosstab(
    pd.Series(
        original_predictions,
        name="Original prediction",
    ),
    pd.Series(
        intervention_predictions[
            "candidate_mentions_masked"
        ],
        name="Candidate-masked prediction",
    ),
)

candidate_transition_counts = candidate_transition_counts.reindex(
    index=LABEL_ORDER,
    columns=LABEL_ORDER,
    fill_value=0,
)

candidate_transition_counts

Candidate-masked prediction,Against,Favor,Neither
Original prediction,,,
Against,267,2,182
Favor,7,63,156
Neither,0,0,213


In [ ]:
candidate_transition_rates = (
    candidate_transition_counts
    .div(
        candidate_transition_counts.sum(axis=1),
        axis=0,
    )
)

candidate_transition_rates.round(3)

Candidate-masked prediction,Against,Favor,Neither
Original prediction,,,
Against,0.592,0.004,0.404
Favor,0.031,0.279,0.690
Neither,0.000,0.000,1.000


Candidate-mention masking produces a strong shift toward the **Neither** class. Only 59.2% of originally predicted Against cases and 27.9% of originally predicted Favor cases retain their original prediction, while 40.4% and 69.0%, respectively, switch to Neither. In contrast, all originally predicted Neither cases remain unchanged. This suggests that explicit candidate references provide important evidence for directional stance predictions, whereas removing them causes RoBERTa to fall back disproportionately to the neutral class.

---

### **18. Save Evaluation Results**

Save the main RoBERTa intervention results for subsequent reporting and visualization.

In [ ]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

comparison_results.to_csv(
    RESULTS_DIR / "overall_intervention_results.csv",
    index=False,
)

control_comparison.to_csv(
    RESULTS_DIR / "matched_control_results.csv",
    index=False,
)

target_results.to_csv(
    RESULTS_DIR / "target_specific_results.csv",
    index=False,
)

class_results.to_csv(
    RESULTS_DIR / "class_specific_results.csv",
    index=False,
)

coverage_results.to_csv(
    RESULTS_DIR / "intervention_coverage.csv",
    index=False,
)

conditional_control_comparison.to_csv(
    RESULTS_DIR / "conditional_control_results.csv",
    index=False,
)

transition_results.to_csv(
    RESULTS_DIR / "correctness_transitions.csv",
    index=False,
)

candidate_transition_counts.to_csv(
    RESULTS_DIR / "candidate_prediction_transitions.csv"
)

probability_results.to_csv(
    RESULTS_DIR / "probability_change_results.csv",
    index=False,
)

probability_control_comparison.to_csv(
    RESULTS_DIR / "probability_control_results.csv",
    index=False,
)

affected_results.to_csv(
    RESULTS_DIR / "affected_example_results.csv",
    index=False,
)

print(f"Saved RoBERTa intervention results to: {RESULTS_DIR}")

Saved RoBERTa intervention results to: results/roberta_interventions


---

### **19. Summary of Main Findings**

The intervention analysis reveals clear differences in how strongly the RoBERTa classifier relies on the tested information sources.

- **The supplied target has a substantial influence on the model.** Masking the target decreases Macro-F1 from 0.8123 to 0.7422 and changes 11.3% of predictions. Swapping the target while keeping the retrieved posts unchanged produces a much stronger prediction flip rate of 71.6%, showing that RoBERTa is highly sensitive to the supplied target identity. Because the swapped inputs do not have valid gold labels, this intervention is interpreted only as a behavioral sensitivity test. Target-specific results further show that swapping affects Trump-target predictions particularly strongly, with a flip rate of 91.5%.

- **Explicit candidate references have the strongest performance effect.** Masking candidate mentions reduces Macro-F1 to 0.5542 and changes 39.0% of all predictions. Among the 843 examples whose input is actually modified, the flip rate reaches 41.2% and the probability assigned to the originally predicted class decreases by 0.355 on average. Matched random-control removals change only 3.8% of affected predictions on average, indicating that the observed effect is substantially larger than the general effect of removing comparable amounts of text.

- **The candidate-reference effect is particularly strong for directional stance predictions.** The Favor class F1 decreases from 0.785 to 0.380 after candidate masking, mainly because recall falls substantially. In the descriptive transition analysis, 69.0% of originally predicted Favor cases and 40.4% of originally predicted Against cases switch to Neither, while all originally predicted Neither cases remain unchanged. This suggests that explicit candidate references provide important evidence for directional stance decisions and that removing them causes the model to fall back disproportionately to the neutral class.

- **The selected label-correlated lexical cues show little evidence of strong model reliance.** The Top-10 and Top-25 cue interventions affect only 14.2% and 30.9% of test examples, respectively, and produce almost no change in overall Macro-F1 or prediction behavior. Even when the analysis is restricted to affected examples, their effects remain small and close to those produced by matched random controls.

Overall, RoBERTa does not rely equally on all tested information sources. Its predictions are strongly sensitive to the supplied target and especially to explicit candidate references in the retrieved context posts, while the selected label-correlated lexical cues contribute comparatively little. The results therefore suggest that explicit target-related information plays a central role in RoBERTa's stance decisions, although the strong dependence on candidate mentions also raises the question of how well the model can infer stance from more indirect contextual evidence.